# Governança de Matéria-Prima — Base de Dados

**Este notebook não é dashboard nem análise. É a camada de estruturação da base.**

O produto final são tabelas planas, com grão declarado e contrato de colunas estável, prontas
para serem consumidas por outras plataformas: a camada de extração para Google Sheets pluga no
bloco final, e o destino é servir de base governável para o Lovable.

## Escopo

**Fabric governance puro: MP, tecido, artigo, malharia.

| Tabela | Grão (PK) | Conteúdo |
|---|---|---|
| `dim_mp_fornecimento` | `fabric_sku_id` | MP, tecido, artigo, malharia, status, custo unitário, MOQ, frete, acordo de pagamento, **LT malharia cadastrado** |
| `fact_sku_bom` | `sku` × `fabric_id` | ficha técnica real, consumo por peça, custo de MP |
| `fact_mp_lt_realizado` | `fabric_order_id` | **LT malharia realizado**, aderência vs data original e vs repactuada |
| `fact_sku_economics` | `sku` | CMV, markup, custo de MP, % MP no CMV |
| `mart_produto_mp` | `product_name` | derivada comercial: recorte de venda L8M, tecido principal |
| `dicionario_dados` | `tabela` × `coluna` | metadado legível por máquina |
| `df_governanca` | `check` | auditoria de qualidade do cadastro |

## Convenções de contrato

- `snake_case`, sem acento, sem espaço.
- Só tipos escalares. **Nada de `ARRAY`/`STRUCT`** — Sheets e Lovable não ingerem. Lista de
  malharias sai como string `;`-separada, com `n_malharias` ao lado.
- Datas em ISO `YYYY-MM-DD`. Toda tabela carrega `atualizado_em` como última coluna.
- Todo `*_id` preservado, para permitir modelagem relacional no destino.
- **NULL significa ausência real no cadastro.** Nunca `0` nem `"N/A"`.
- Ordem de colunas fixa, declarada em `SCHEMA_CONTRATO`.

## Achados que corrigem a documentação anterior

1. **`sku_bill_of_materials` não é fonte concorrente.** Existe em `integrated` (a documentação
   só conhecia a versão `_br` inacessível). As 24.388 linhas de `muninn_product_skus_fabrics`
   casam 100% em `(sku, fabric_id, consumption)` — **zero divergência de consumo**. Com 36.139
   linhas, é superset. `muninn_product_skus_fabrics` segue como sistema de registro.
2. **Gramatura e largura não existem no data lake.** Varredura de `INFORMATION_SCHEMA` em
   `integrated` + `sop_silver` não retorna nenhuma coluna de MP, e os nomes de artigo
   ("Modal", "Bi Stretch") não codificam a informação. As colunas existem no contrato com
   valor nulo, para o consumidor não quebrar quando a fonte aparecer.
3. **Tecido NÃO é entidade genérica.** `product_color_id` está preenchido em **100%** dos 602
   tecidos. A hierarquia "Artigo → Tecido → Fabric SKU" da documentação anterior está
   incompleta: falta a dimensão de cor no meio.
4. **`muninn_fabric_orders` tem repactuação de data**, igual ao lead time produtivo:
   `estimated_invoicing_date` é o compromisso original e `updated_invoicing_date` o repactuado.
   A métrica de contrato usa a **original** — medir só contra a repactuada esconde atraso,
   porque a data se move junto com ele.

## Limitações que o consumidor precisa conhecer

- **Base é snapshot, não série histórica.** Nenhuma tabela Muninn versiona preço/status
  (`ingestion_date` é metadado de carga). Só `cmv_model` tem eixo temporal real. **Quem
  construir tendência de custo de MP em cima disto vai errar.**
- **Custo de MP ≠ CMV.** `custo_mp_*` é matéria-prima isolada; `cmv_unitario` é custo total.
  Divergência entre os dois não é erro — é escopo diferente.
- **LT malharia cadastrado tem cobertura parcial.** Onde não há cadastro, fica em branco.


In [ ]:
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# --- Parâmetros da base --------------------------------------------------------------------
STATUS_MP_VIGENTE = "available"      # status de cotação considerado vigente
JANELA_VENDAS_MESES = 8              # janela do recorte comercial (mart_produto_mp)

# Exclusões de linha do recorte comercial (herdadas da query de produção).
EXCLUSOES_PRODUTO = ["%ziraldo%", "% xp%", "%maluquinho%", "% b2b %"]

# --- Flags de saída ------------------------------------------------------------------------
EXPORTAR_CSV = True
ESCREVER_SHEETS = True              # ponto de plugue da camada de extração para Sheets

DIR_EXPORT = "exports"
DATA_REFERENCIA = datetime.now(timezone.utc).date()
ATUALIZADO_EM = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

# --- Baselines de auditoria ----------------------------------------------------------------
# Medidos em 2026-08-07. Servem de referência para detectar degradação de cadastro,
# não como valor esperado imutável — o cadastro muda legitimamente ao longo do tempo.
BASELINE = {
    "dim_mp_fornecimento_linhas": 401,
    "fact_sku_bom_linhas": 24388,
    "sku_bom_derivado_linhas": 36139,
    "reconciliacao_consumo_divergente": 0,   # este é obrigatório: divergência = falha
    "fabrics_total": 602,
    "fabrics_com_cor": 602,                  # 100% dos tecidos amarrados a cor
    "artigos_total": 96,
    "mart_produtos": 166,
}

print(f"Data de referência : {DATA_REFERENCIA}")
print(f"Atualizado em      : {ATUALIZADO_EM}")
print(f"Exportar CSV       : {EXPORTAR_CSV}")
print(f"Escrever Sheets    : {ESCREVER_SHEETS}")


## 1. `dim_mp_fornecimento` — grão: `fabric_sku_id`

Tabela central da base: uma linha por cotação de tecido numa malharia.

Duas escolhas deliberadas, diferentes do padrão de query anterior:

1. **Não filtra `status = 'available'`.** A cotação desativada permanece como histórico
   auditável, com `status_cotacao` exposto. Só a agregação de custo vigente
   (`custo_min_fabric` / `custo_max_fabric` / `n_malharias`) considera as vigentes.
2. **`supplier_id` vem sempre de `muninn_knitting_factories`**, nunca de
   `muninn_apparel_manufacturers`. Os dois papéis apontam para a mesma `muninn_suppliers` —
   é o erro mais fácil de cometer aqui e o bloco de auditoria testa isso explicitamente.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

dim_mp_fornecimento = _dntk.execute_sql(
  '-- dim_mp_fornecimento — GRÃO: fabric_sku_id (1 linha por cotação de tecido numa malharia)\n-- Fonte de verdade de: MP/tecido/artigo, malharia, status, custo unitário, LT malharia cadastrado.\n-- NÃO filtra status=\'available\': mantém a cotação desativada como histórico auditável.\n-- Só a agregação de custo vigente (custo_min/max_fabric) considera \'available\'.\nWITH acordo_fornecedor AS (\n    -- muninn_supplier_agreement tem supplier_agreement_id duplicado (137 linhas / 122 ids) -> dedupe\n    SELECT\n        supplier_agreement_id,\n        ANY_VALUE(agreement_status)          AS agreement_status,\n        ANY_VALUE(agreement_start_date)      AS agreement_start_date,\n        ANY_VALUE(agreement_expiration_date) AS agreement_expiration_date\n    FROM `insider-data-lake.integrated.muninn_supplier_agreement`\n    GROUP BY supplier_agreement_id\n),\n\nlt_malharia_cadastrado AS (\n    -- LT cadastrado da malharia por artigo x malharia. Cobertura parcial:\n    -- LEFT JOIN proposital lá embaixo -> onde não há cadastro, fica NULL (em branco).\n    SELECT\n        article_id,\n        knitting_factory_id,\n        coloring_time                   AS lt_tingimento_cadastrado_dias,\n        production_time                 AS lt_producao_cadastrado_dias,\n        coloring_time + production_time AS lt_malharia_cadastrado_dias,\n        mininimum_order_volume          AS volume_minimo_pedido_artigo,\n        multiple_volume_to_coloring     AS multiplo_volume_tingimento,\n        bought_by_supplier              AS comprado_pelo_fornecedor\n    FROM `insider-data-lake.integrated.muninn_articles_knitting_factories`\n),\n\nbase AS (\n    SELECT\n        mfs.id                        AS fabric_sku_id,\n        mfs.sku                       AS fabric_sku,\n        mfs.fabric_id,\n        mf.name                       AS fabric_name,\n        mf.article_id,\n        ma.name                       AS article_name,\n        ma.unit                       AS article_unit,\n\n        -- Papel de fornecedor: SEMPRE malharia (muninn_knitting_factories).\n        -- Nunca muninn_apparel_manufacturers, que aponta para a mesma muninn_suppliers.\n        mfs.knitting_factory_id,\n        mkf.supplier_id               AS malharia_supplier_id,\n        ms.alias                      AS malharia_nome,\n        ms.legal_name                 AS malharia_razao_social,\n        ms.tax_identification_code    AS malharia_cnpj,\n        ms.type                       AS malharia_tipo,\n        ms.city                       AS malharia_cidade,\n        ms.state                      AS malharia_uf,\n        mkf.freight_type              AS tipo_frete,\n\n        mfs.status                    AS status_cotacao,\n        mfs.unit_price                AS custo_unitario,\n        mfs.minimum_volume_per_order  AS volume_minimo_pedido,\n        mfs.multiple_volume_per_order AS multiplo_volume_pedido,\n        mfs.volume_per_roll           AS volume_por_rolo,\n        mfs.factory_color_code        AS codigo_cor_fabrica,\n        mfs.invoice_fabric_code       AS codigo_tecido_nf,\n        mfs.invoice_fabric_name       AS nome_tecido_nf,\n\n        mfs.payment_agreement_id,\n        mpa.label                     AS acordo_pagamento_label,\n        mpa.payment_method            AS forma_pagamento,\n        mpa.installments              AS parcelas,\n        mpa.first_payment_due_days    AS dias_primeiro_vencimento,\n        af.agreement_status           AS status_acordo_fornecedor,\n\n        lt.lt_tingimento_cadastrado_dias,\n        lt.lt_producao_cadastrado_dias,\n        lt.lt_malharia_cadastrado_dias,\n        lt.volume_minimo_pedido_artigo,\n        lt.multiplo_volume_tingimento,\n        lt.comprado_pelo_fornecedor,\n\n        -- GAP DECLARADO: não existe fonte para gramatura/largura no data lake (verificado em\n        -- INFORMATION_SCHEMA de integrated + sop_silver em 2026-08-07). A coluna existe no\n        -- contrato para o consumidor não quebrar quando a fonte aparecer.\n        CAST(NULL AS FLOAT64)         AS gramatura_g_m2,\n        CAST(NULL AS FLOAT64)         AS largura_cm,\n\n        -- product_color_id está preenchido em 100% dos 602 tecidos (medido em 2026-08-07).\n        -- Ou seja: tecido NÃO é entidade genérica de cadastro — é sempre amarrado a uma cor de\n        -- produto. A hierarquia "Artigo -> Tecido -> Fabric SKU" da documentação anterior está\n        -- incompleta: falta a dimensão de cor no meio.\n        mf.product_color_id,\n        mf.product_color_id IS NOT NULL AS fabric_amarrado_a_cor,\n\n        -- field_errors vem como JSON string; \'{}\' significa cadastro OK (108 de 124 fornecedores).\n        ms.field_errors               AS malharia_field_errors,\n        (ms.field_errors IS NOT NULL AND ms.field_errors != \'{}\') AS malharia_cadastro_incompleto,\n\n        mfs.update_source             AS origem_ultima_alteracao,\n        mfs.update_author_id          AS autor_ultima_alteracao,\n        DATE(mfs.created_at)          AS cotacao_criada_em,\n        DATE(mfs.updated_at)          AS cotacao_atualizada_em\n    FROM `insider-data-lake.integrated.muninn_fabric_skus`               AS mfs\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics`              AS mf  ON mf.id  = mfs.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles`             AS ma  ON ma.id  = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories`   AS mkf ON mkf.id = mfs.knitting_factory_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers`            AS ms  ON ms.id  = mkf.supplier_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_payment_agreement`    AS mpa ON mpa.id = mfs.payment_agreement_id\n    LEFT JOIN acordo_fornecedor      AS af ON af.supplier_agreement_id = ms.current_supplier_agreement_id\n    LEFT JOIN lt_malharia_cadastrado AS lt ON lt.article_id = mf.article_id\n                                          AND lt.knitting_factory_id = mfs.knitting_factory_id\n),\n\n-- Faixa de custo e concentração de fornecimento por tecido, só sobre cotação vigente.\n-- CTE agregada (não função de janela): COUNT(DISTINCT) não é permitido com OVER no BigQuery.\nfaixa_custo_por_fabric AS (\n    SELECT\n        fabric_id,\n        MIN(custo_unitario)                 AS custo_min_fabric,\n        MAX(custo_unitario)                 AS custo_max_fabric,\n        COUNT(DISTINCT knitting_factory_id) AS n_malharias,\n        -- Saída plana: string \';\'-separada, nunca ARRAY (Sheets/Lovable não ingerem ARRAY).\n        STRING_AGG(DISTINCT malharia_nome, \'; \' ORDER BY malharia_nome) AS malharias_nomes\n    FROM base\n    WHERE status_cotacao = \'available\'\n    GROUP BY fabric_id\n)\n\nSELECT\n    b.*,\n    f.custo_min_fabric,\n    f.custo_max_fabric,\n    IFNULL(f.n_malharias, 0) AS n_malharias,\n    f.malharias_nomes\nFROM base AS b\nLEFT JOIN faixa_custo_por_fabric AS f ON f.fabric_id = b.fabric_id\nORDER BY b.article_name, b.fabric_name, b.fabric_sku_id',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
dim_mp_fornecimento

## 2. `fact_sku_bom` — grão: `sku` × `fabric_id`

Ficha técnica real: uma linha por tecido que compõe o SKU.

**Sistema de registro é `muninn_product_skus_fabrics`.** A tabela `sku_bill_of_materials` entra
apenas como flag de reconciliação (`presente_em_sku_bom`) — comparação executada em 2026-08-07
mostrou que as 24.388 linhas casam 100% em `(sku, fabric_id, consumption)`, com **zero
divergência de consumo**, e que a `sku_bill_of_materials` é superset (36.139 linhas).

**O campo de consumo se chama `consumo_sku`, não `consumption`.** É deliberado:
`muninn_products_articles` usa o mesmo nome de campo na origem mas vive em grão de **produto**.
Nomes diferentes impedem que alguém some estimativa de produto com consumo real de SKU.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

fact_sku_bom = _dntk.execute_sql(
  '-- fact_sku_bom — GRÃO: sku x fabric_id (ficha técnica real, 1 linha por tecido do SKU)\n-- Sistema de registro: muninn_product_skus_fabrics.\n-- sku_bill_of_materials entra SÓ como flag de reconciliação (é superset, não fonte concorrente).\nWITH faixa_custo_por_fabric AS (\n    SELECT\n        mfs.fabric_id,\n        MIN(mfs.unit_price)                     AS custo_min_fabric,\n        MAX(mfs.unit_price)                     AS custo_max_fabric,\n        COUNT(DISTINCT mfs.knitting_factory_id) AS n_malharias,\n        STRING_AGG(DISTINCT ms.alias, \'; \' ORDER BY ms.alias) AS malharias_nomes\n    FROM `insider-data-lake.integrated.muninn_fabric_skus`              AS mfs\n    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories`  AS mkf ON mkf.id = mfs.knitting_factory_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers`           AS ms  ON ms.id  = mkf.supplier_id\n    WHERE mfs.status = \'available\'\n    GROUP BY mfs.fabric_id\n),\n\n-- Reconciliação: chave (sku, fabric_id) presente na tabela derivada sku_bill_of_materials.\nbom_derivado AS (\n    SELECT DISTINCT sku, fabric_id\n    FROM `insider-data-lake.integrated.sku_bill_of_materials`\n)\n\nSELECT\n    mps.sku,\n    mps.sku_name,\n    mpsf.product_sku_id,\n    mps.product_id,\n    mp.product_name,\n\n    s.sku_state,\n    s.gender,\n    s.color,\n    s.size,\n    s.category,\n    s.family,\n    mps.product_state_id,\n\n    mpsf.fabric_id,\n    mf.name                       AS fabric_name,\n    mf.article_id,\n    ma.name                       AS article_name,\n    ma.unit                       AS article_unit,\n\n    -- NOME DELIBERADO: consumo_sku, nunca \'consumption\'.\n    -- Impede que alguém some isto com o consumo estimado de muninn_products_articles,\n    -- que vive em grão de PRODUTO e usa o mesmo nome de campo na origem.\n    mpsf.consumption              AS consumo_sku,\n\n    fc.custo_min_fabric           AS custo_unitario_min_fabric,\n    fc.custo_max_fabric           AS custo_unitario_max_fabric,\n    fc.n_malharias,\n    fc.malharias_nomes,\n\n    -- Custo de MP deste tecido no SKU = consumo x custo unitário do tecido.\n    fc.custo_min_fabric * mpsf.consumption AS custo_mp_min,\n    fc.custo_max_fabric * mpsf.consumption AS custo_mp_max,\n\n    -- Custo de MP do SKU já calculado por outro processo (integrated.skus.fabric_cost).\n    -- Fica lado a lado para reconciliação — o escopo pode não ser idêntico ao calculado aqui.\n    s.fabric_cost                 AS custo_mp_referencia_skus,\n\n    bd.sku IS NOT NULL            AS presente_em_sku_bom\nFROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\nLEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\nLEFT JOIN `insider-data-lake.integrated.muninn_products`     AS mp  ON mp.product_id = mps.product_id\nLEFT JOIN `insider-data-lake.integrated.muninn_fabrics`      AS mf  ON mf.id = mpsf.fabric_id\nLEFT JOIN `insider-data-lake.integrated.muninn_articles`     AS ma  ON ma.id = mf.article_id\nLEFT JOIN `insider-data-lake.integrated.skus`                AS s   ON s.sku = mps.sku\nLEFT JOIN faixa_custo_por_fabric AS fc ON fc.fabric_id = mpsf.fabric_id\nLEFT JOIN bom_derivado           AS bd ON bd.sku = mps.sku AND bd.fabric_id = mpsf.fabric_id\nORDER BY mps.sku, mpsf.fabric_id',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
fact_sku_bom

## 3. `fact_mp_lt_realizado` — grão: `fabric_order_id`

Lead time realizado da malharia: uma linha por pedido de tecido. **É o único "realizado" do
escopo** — confecção está fora por decisão.

### Original vs repactuado

`muninn_fabric_orders` guarda duas datas de compromisso:

| Coluna | Significado | Papel |
|---|---|---|
| `estimated_invoicing_date` | compromisso **original** | **métrica de contrato** |
| `updated_invoicing_date` | compromisso **repactuado** | diagnóstico |

Medir aderência só contra a data repactuada **esconde atraso**, porque a data se move junto com
ele. É o mesmo padrão da correção de postergação já validada no lead time produtivo. Por isso
`atraso_vs_original_dias` é a métrica oficial e `atraso_vs_repactuado_dias` / `dias_repactuacao`
ficam como diagnóstico ao lado.

O `lt_malharia_cadastrado_dias` vem junto, no mesmo grão, para permitir a leitura
cadastrado vs realizado sem precisar de outro join.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

fact_mp_lt_realizado = _dntk.execute_sql(
  '-- fact_mp_lt_realizado — GRÃO: fabric_order_id (1 linha por pedido de tecido à malharia)\n-- Único "realizado" do escopo. Confecção está fora por decisão.\n--\n-- Padrão original vs repactuado (espelha a correção de postergação do lead time produtivo):\n--   estimated_invoicing_date = compromisso ORIGINAL   -> métrica de contrato\n--   updated_invoicing_date   = compromisso REPACTUADO -> diagnóstico\n-- Medir só contra o repactuado esconde atraso, porque a data se move junto com o atraso.\nWITH lt_malharia_cadastrado AS (\n    SELECT\n        article_id,\n        knitting_factory_id,\n        coloring_time + production_time AS lt_malharia_cadastrado_dias\n    FROM `insider-data-lake.integrated.muninn_articles_knitting_factories`\n),\n\npedidos AS (\n    SELECT\n        mfo.id                        AS fabric_order_id,\n        mfo.order_code                AS fabric_order_code,\n        mfo.fabric_sku_id,\n        mfo.knitting_factory_id,\n        mfo.article_order_id,\n        mfo.status                    AS status_pedido,\n        mfo.volume                    AS volume_pedido,\n        mfo.invoiced_volume           AS volume_faturado,\n        mfo.available_volume          AS volume_disponivel,\n        mfo.price                     AS preco_pedido,\n        mfo.invoiced_price            AS preco_faturado,\n        DATE(mfo.created_at)          AS data_pedido,\n        mfo.estimated_invoicing_date  AS data_faturamento_estimada_original,\n        mfo.updated_invoicing_date    AS data_faturamento_repactuada,\n        mfo.real_invoicing_date       AS data_faturamento_real\n    FROM `insider-data-lake.integrated.muninn_fabric_orders` AS mfo\n    WHERE mfo.deleted_at IS NULL\n)\n\nSELECT\n    p.fabric_order_id,\n    p.fabric_order_code,\n    p.fabric_sku_id,\n    mfs.sku                       AS fabric_sku,\n    mfs.fabric_id,\n    mf.name                       AS fabric_name,\n    mf.article_id,\n    ma.name                       AS article_name,\n    ma.unit                       AS article_unit,\n\n    p.knitting_factory_id,\n    ms.alias                      AS malharia_nome,\n    ms.legal_name                 AS malharia_razao_social,\n\n    p.status_pedido,\n    p.volume_pedido,\n    p.volume_faturado,\n    p.volume_disponivel,\n    p.preco_pedido,\n    p.preco_faturado,\n\n    p.data_pedido,\n    p.data_faturamento_estimada_original,\n    p.data_faturamento_repactuada,\n    p.data_faturamento_real,\n\n    -- LT realizado: do pedido até o faturamento real.\n    DATE_DIFF(p.data_faturamento_real, p.data_pedido, DAY)                        AS lt_malharia_realizado_dias,\n    -- LT prometido originalmente, no mesmo eixo.\n    DATE_DIFF(p.data_faturamento_estimada_original, p.data_pedido, DAY)           AS lt_malharia_estimado_original_dias,\n    -- MÉTRICA DE CONTRATO: aderência contra o compromisso original.\n    DATE_DIFF(p.data_faturamento_real, p.data_faturamento_estimada_original, DAY) AS atraso_vs_original_dias,\n    -- Diagnóstico: aderência contra o compromisso já repactuado.\n    DATE_DIFF(p.data_faturamento_real, p.data_faturamento_repactuada, DAY)        AS atraso_vs_repactuado_dias,\n    -- Diagnóstico: quanto a data foi empurrada (equivalente à postergação do LT produtivo).\n    DATE_DIFF(p.data_faturamento_repactuada, p.data_faturamento_estimada_original, DAY) AS dias_repactuacao,\n\n    -- LT cadastrado da malharia para o artigo, no mesmo grão -> cadastrado vs realizado.\n    lt.lt_malharia_cadastrado_dias,\n    DATE_DIFF(p.data_faturamento_real, p.data_pedido, DAY) - lt.lt_malharia_cadastrado_dias\n                                                                                 AS desvio_vs_cadastrado_dias,\n\n    CASE\n        WHEN p.data_faturamento_estimada_original IS NULL THEN \'MISSING_ESTIMATED\'\n        WHEN p.data_faturamento_real IS NULL              THEN \'PENDENTE\'\n        WHEN p.data_faturamento_real > p.data_faturamento_estimada_original THEN \'LATE\'\n        WHEN p.data_faturamento_real = p.data_faturamento_estimada_original THEN \'ON_TIME\'\n        ELSE \'EARLY\'\n    END AS status_faturamento_mp\nFROM pedidos AS p\nLEFT JOIN `insider-data-lake.integrated.muninn_fabric_skus`        AS mfs ON mfs.id = p.fabric_sku_id\nLEFT JOIN `insider-data-lake.integrated.muninn_fabrics`            AS mf  ON mf.id  = mfs.fabric_id\nLEFT JOIN `insider-data-lake.integrated.muninn_articles`           AS ma  ON ma.id  = mf.article_id\nLEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = p.knitting_factory_id\nLEFT JOIN `insider-data-lake.integrated.muninn_suppliers`          AS ms  ON ms.id  = mkf.supplier_id\nLEFT JOIN lt_malharia_cadastrado AS lt ON lt.article_id = mf.article_id\n                                      AND lt.knitting_factory_id = p.knitting_factory_id\nORDER BY p.data_pedido DESC, p.fabric_order_id',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
fact_mp_lt_realizado

## 4. `fact_sku_economics` — grão: `sku`

CMV, markup e custo de MP lado a lado.

> ### ⚠️ Aviso de escopo — leia antes de usar qualquer número daqui
>
> **`custo_mp_*` é matéria-prima isolada. `cmv_unitario` é custo total do SKU.**
>
> Divergência entre os dois **não é erro** — são escopos diferentes. Por isso esta tabela expõe
> as fontes alternativas de custo lado a lado (`custo_mp_referencia_skus`,
> `custo_variante_unitario`, `custo_sku_muninn`, `custo_industrializacao`) em vez de eleger uma
> só e esconder as outras. Quem consumir só um número, sem este contexto, vai concluir errado.

Deduplicações necessárias, medidas em 2026-08-07:

- `markup_prices` tem 69.302 linhas para 4.806 SKUs (histórico de vigência) → filtra vigente e
  fica com a linha de `valid_from` mais recente.
- `cmv_model` é diário → fica com a última `date` por SKU, exposta em `cmv_data_ref`.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

fact_sku_economics = _dntk.execute_sql(
  '-- fact_sku_economics — GRÃO: sku (1 linha por SKU com ficha técnica de tecido)\n--\n-- AVISO DE ESCOPO (crítico para quem consome só o número):\n-- custo_mp_* é MATÉRIA-PRIMA ISOLADA. cmv_unitario é CUSTO TOTAL do SKU.\n-- Divergência entre os dois NÃO é erro — são escopos diferentes. Por isso a base expõe\n-- as fontes alternativas de custo lado a lado em vez de eleger uma só.\nWITH custo_mp_por_sku AS (\n    -- Soma o custo de MP de TODOS os tecidos do SKU (um SKU pode ter vários).\n    SELECT\n        mps.sku,\n        COUNT(DISTINCT mpsf.fabric_id)              AS n_tecidos_no_sku,\n        SUM(mpsf.consumption)                       AS consumo_total_sku,\n        SUM(fc.custo_min_fabric * mpsf.consumption) AS custo_mp_min,\n        SUM(fc.custo_max_fabric * mpsf.consumption) AS custo_mp_max\n    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\n    JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\n    LEFT JOIN (\n        SELECT fabric_id, MIN(unit_price) AS custo_min_fabric, MAX(unit_price) AS custo_max_fabric\n        FROM `insider-data-lake.integrated.muninn_fabric_skus`\n        WHERE status = \'available\'\n        GROUP BY fabric_id\n    ) AS fc ON fc.fabric_id = mpsf.fabric_id\n    GROUP BY mps.sku\n),\n\ncmv_vigente AS (\n    -- cmv_model é diário; pega a última data disponível por SKU.\n    SELECT\n        sku,\n        date                  AS cmv_data_ref,\n        unitary_cost          AS cmv_unitario,\n        unitary_cost_tax_free AS cmv_unitario_sem_imposto,\n        sold_quantity         AS cmv_qtd_vendida_ref\n    FROM `insider-data-lake.integrated.cmv_model`\n    QUALIFY ROW_NUMBER() OVER (PARTITION BY sku ORDER BY date DESC) = 1\n),\n\nmarkup_vigente AS (\n    -- markup_prices tem 69.302 linhas para 4.806 SKUs (histórico de vigência) -> dedupe\n    -- para a linha vigente mais recente.\n    SELECT\n        sku,\n        markup_price,\n        DATE(valid_from) AS markup_valid_from\n    FROM `insider-data-lake.integrated.markup_prices`\n    WHERE valid_to IS NULL OR valid_to > CURRENT_TIMESTAMP()\n    QUALIFY ROW_NUMBER() OVER (PARTITION BY sku ORDER BY valid_from DESC) = 1\n),\n\ncusto_variante AS (\n    SELECT sku, ANY_VALUE(unit_variant_cost) AS custo_variante_unitario\n    FROM `insider-data-lake.integrated.product_cost`\n    GROUP BY sku\n)\n\nSELECT\n    s.sku,\n    s.sku_name,\n    s.product_id,\n    s.product_name,\n    s.sku_state,\n    s.gender,\n    s.color,\n    s.size,\n    s.category,\n    s.family,\n\n    -- Matéria-prima (escopo: MP isolada)\n    mp.n_tecidos_no_sku,\n    mp.consumo_total_sku,\n    mp.custo_mp_min,\n    mp.custo_mp_max,\n    s.fabric_cost                 AS custo_mp_referencia_skus,\n\n    -- CMV (escopo: custo total)\n    c.cmv_data_ref,\n    c.cmv_unitario,\n    c.cmv_unitario_sem_imposto,\n    c.cmv_qtd_vendida_ref,\n\n    -- Preço e markup\n    mk.markup_price,\n    mk.markup_valid_from,\n    s.sku_price,\n    s.full_price,\n    SAFE_DIVIDE(mk.markup_price, c.cmv_unitario_sem_imposto) AS markup_ratio,\n\n    -- Peso da MP dentro do custo total. Fora de (0, 1] indica escopo de custo trocado.\n    SAFE_DIVIDE(mp.custo_mp_min, c.cmv_unitario) AS pct_mp_no_cmv,\n\n    -- Fontes alternativas de custo, para auditoria de escopo (não são a mesma coisa).\n    cv.custo_variante_unitario,\n    mps.sku_cost                         AS custo_sku_muninn,\n    s.industrialization_cost             AS custo_industrializacao,\n    s.industrialized_sku_cost            AS custo_sku_industrializado,\n    s.manufacturer_finished_product_cost AS custo_produto_acabado_fornecedor\nFROM `insider-data-lake.integrated.skus` AS s\nLEFT JOIN custo_mp_por_sku AS mp ON mp.sku = s.sku\nLEFT JOIN cmv_vigente      AS c  ON c.sku  = s.sku\nLEFT JOIN markup_vigente   AS mk ON mk.sku = s.sku\nLEFT JOIN custo_variante   AS cv ON cv.sku = s.sku\nLEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.sku = s.sku\n-- Base de MP: só SKUs que têm ficha técnica de tecido.\nWHERE mp.sku IS NOT NULL\nORDER BY s.product_name, s.sku',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
fact_sku_economics

## 5. `mart_produto_mp` — grão: `product_name`

Camada **comercial derivada**, não a base. As tabelas 1–4 mantêm o universo completo de
cadastro; este mart aplica o recorte da query de produção em uso:

- produtos com venda nos últimos 8 meses (`fpa.analytical_dre`)
- status do produto por prioridade de `sku_state`, restrito a
  `ativo_perene` / `ativo_em_lancamento` / `desativado`
- exclusões de linha: `ziraldo`, ` xp`, `maluquinho`, ` b2b `
- normalização `Modal (\d+)` → `Modal`
- tecido principal = maior consumo mediano por produto (`QUALIFY ROW_NUMBER()`)

**Detalhe que muda o resultado:** a normalização funde vários `article_id` sob um mesmo nome, o
que torna qualquer `article_id` do grupo arbitrário. Por isso os lead times são chaveados pelo
**nome normalizado do artigo**, não por id, e `article_id` não é exposto nesta tabela.

Colunas acrescentadas em relação à query original: malharias, custo unitário e por peça, lead
time da malharia (cadastrado e realizado), % de pedidos de MP atrasados, CMV, markup.

> Este bloco lê `fpa.analytical_dre` e `cmv_model` — é o mais caro do notebook
> (~5 GB por execução). Os demais varrem poucos MB.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

mart_produto_mp = _dntk.execute_sql(
  '-- mart_produto_mp — GRÃO: product_name (1 linha por produto)\n-- Camada COMERCIAL derivada. Reproduz fielmente o recorte da query de produção\n-- (venda nos últimos 8 meses, exclusões de linha, tecido principal por consumo mediano)\n-- e acrescenta as colunas de governança que a query original não trazia.\n--\n-- As tabelas governadas (dim_mp_fornecimento, fact_sku_bom, fact_sku_economics) mantêm o\n-- universo COMPLETO de cadastro. Este mart é o recorte comercial, não a base.\nWITH fabric_costs AS (\n    SELECT\n        mfs.id AS fabric_sku_id,\n        mfs.fabric_id,\n        mfs.knitting_factory_id,\n        mfs.unit_price,\n        mfs.minimum_volume_per_order,\n        mf.name  AS fabric_name,\n        mf.article_id,\n        ma.name  AS article_name,\n        ma.unit  AS article_unit,\n        ms.alias AS knitting_factory_name\n    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics`            AS mf  ON mf.id  = mfs.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles`           AS ma  ON ma.id  = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers`          AS ms  ON ms.id  = mkf.supplier_id\n    WHERE mfs.status = \'available\'\n),\n\nfabric_min_max_cost AS (\n    SELECT\n        fc.fabric_id,\n        fc.fabric_name,\n        MIN(fc.unit_price)                     AS min_fabric_cost,\n        MAX(fc.unit_price)                     AS max_fabric_cost,\n        MIN(fc.minimum_volume_per_order)       AS minimum_volume_per_order,\n        COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,\n        -- Saída plana: STRING_AGG em vez do ARRAY_AGG original (Sheets/Lovable não ingerem ARRAY).\n        STRING_AGG(DISTINCT fc.knitting_factory_name, \'; \' ORDER BY fc.knitting_factory_name) AS knitting_factories_names\n    FROM fabric_costs AS fc\n    GROUP BY fc.fabric_id, fc.fabric_name\n),\n\n-- Nome de artigo normalizado. A query original funde "Modal (5651)", "Modal (5590)" e "Modal"\n-- num único artigo. Como isso agrupa VÁRIOS article_id sob o mesmo nome, o lead time precisa\n-- ser chaveado pelo nome normalizado — chavear por article_id pegaria um id arbitrário do grupo.\nartigo_normalizado AS (\n    SELECT\n        id AS article_id,\n        REGEXP_REPLACE(name, r\'Modal \\(\\d+\\)\', \'Modal\') AS article_name_norm\n    FROM `insider-data-lake.integrated.muninn_articles`\n),\n\nlt_cadastrado_por_artigo AS (\n    SELECT\n        an.article_name_norm,\n        MIN(akf.coloring_time + akf.production_time) AS lt_malharia_cadastrado_min_dias,\n        MAX(akf.coloring_time + akf.production_time) AS lt_malharia_cadastrado_max_dias\n    FROM `insider-data-lake.integrated.muninn_articles_knitting_factories` AS akf\n    JOIN artigo_normalizado AS an ON an.article_id = akf.article_id\n    GROUP BY an.article_name_norm\n),\n\nlt_realizado_por_artigo AS (\n    SELECT\n        an.article_name_norm,\n        COUNT(*) AS n_pedidos_mp_faturados,\n        APPROX_QUANTILES(DATE_DIFF(mfo.real_invoicing_date, DATE(mfo.created_at), DAY), 2)[OFFSET(1)]\n            AS lt_malharia_realizado_mediano_dias,\n        COUNTIF(mfo.real_invoicing_date > mfo.estimated_invoicing_date) AS n_pedidos_mp_atrasados\n    FROM `insider-data-lake.integrated.muninn_fabric_orders` AS mfo\n    JOIN `insider-data-lake.integrated.muninn_fabric_skus`   AS mfs ON mfs.id = mfo.fabric_sku_id\n    JOIN `insider-data-lake.integrated.muninn_fabrics`       AS mf  ON mf.id  = mfs.fabric_id\n    JOIN artigo_normalizado AS an ON an.article_id = mf.article_id\n    WHERE mfo.deleted_at IS NULL\n      AND mfo.real_invoicing_date IS NOT NULL\n    GROUP BY an.article_name_norm\n),\n\nskp_with_sales_l8m AS (\n    SELECT DISTINCT s.product_name\n    FROM `insider-data-lake.fpa.analytical_dre` d\n    LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)\n    WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)\n      AND d.order_status != \'Not authorized\'\n      AND d.quantity > 0\n      AND s.product_name IS NOT NULL\n),\n\nskp_status AS (\n    SELECT\n        s.product_name,\n        CASE\n            WHEN COUNTIF(s.sku_state = \'ativo_perene\') > 0        THEN \'ativo_perene\'\n            WHEN COUNTIF(s.sku_state = \'ativo_em_lancamento\') > 0 THEN \'ativo_em_lancamento\'\n            WHEN COUNTIF(s.sku_state = \'ativo_capsula\') > 0       THEN \'ativo_capsula\'\n            WHEN COUNTIF(s.sku_state = \'personalizacao\') > 0      THEN \'personalizacao\'\n            WHEN COUNTIF(s.sku_state = \'kit\') > 0                 THEN \'kit\'\n            ELSE \'desativado\'\n        END AS product_status\n    FROM `insider-data-lake.integrated.skus` s\n    INNER JOIN skp_with_sales_l8m l8m USING(product_name)\n    GROUP BY s.product_name\n),\n\n-- CMV e markup médios por produto (colunas novas em relação à query original).\neconomics_por_produto AS (\n    SELECT\n        s.product_name,\n        AVG(c.unitary_cost)          AS cmv_unitario_medio,\n        AVG(c.unitary_cost_tax_free) AS cmv_unitario_medio_sem_imposto,\n        AVG(mk.markup_price)         AS markup_price_medio,\n        MAX(c.date)                  AS cmv_data_ref\n    FROM `insider-data-lake.integrated.skus` s\n    LEFT JOIN (\n        SELECT sku, date, unitary_cost, unitary_cost_tax_free\n        FROM `insider-data-lake.integrated.cmv_model`\n        QUALIFY ROW_NUMBER() OVER (PARTITION BY sku ORDER BY date DESC) = 1\n    ) c ON c.sku = s.sku\n    LEFT JOIN (\n        SELECT sku, markup_price\n        FROM `insider-data-lake.integrated.markup_prices`\n        WHERE valid_to IS NULL OR valid_to > CURRENT_TIMESTAMP()\n        QUALIFY ROW_NUMBER() OVER (PARTITION BY sku ORDER BY valid_from DESC) = 1\n    ) mk ON mk.sku = s.sku\n    GROUP BY s.product_name\n),\n\nsku_fabrics AS (\n    SELECT\n        mps.sku,\n        s.product_name,\n        mpsf.fabric_id,\n        mpsf.consumption,\n        fc.min_fabric_cost AS min_fabric_unitary_cost,\n        fc.max_fabric_cost AS max_fabric_unitary_cost,\n        fc.minimum_volume_per_order,\n        ma.unit AS article_unit,\n        ma.name AS article_name,\n        mf.article_id,\n        fc.number_knitting_factories,\n        fc.knitting_factories_names\n    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics`      AS mf  ON mf.id = mpsf.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles`     AS ma  ON ma.id = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\n    LEFT JOIN `insider-data-lake.integrated.skus`                AS s   ON mps.sku = s.sku\n    LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id\n    INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name\n),\n\nproduct_article_agg AS (\n    SELECT\n        sf.product_name,\n        ss.product_status,\n        REGEXP_REPLACE(sf.article_name, r\'Modal \\(\\d+\\)\', \'Modal\') AS article_name,\n        sf.article_unit,\n        -- article_id NÃO é exposto: a normalização funde vários ids num grupo, então qualquer\n        -- id escolhido seria arbitrário. O join de lead time usa o nome normalizado.\n        MIN(sf.minimum_volume_per_order)                       AS minimum_volume_per_order,\n        APPROX_QUANTILES(sf.consumption, 2)[OFFSET(1)]         AS median_article_consumption,\n        MIN(sf.min_fabric_unitary_cost)                        AS custo_unitario_min,\n        MAX(sf.max_fabric_unitary_cost)                        AS custo_unitario_max,\n        MAX(sf.number_knitting_factories)                      AS n_malharias,\n        STRING_AGG(DISTINCT sf.knitting_factories_names, \'; \') AS malharias_nomes\n    FROM sku_fabrics AS sf\n    INNER JOIN skp_status AS ss ON ss.product_name = sf.product_name\n    WHERE ss.product_status IN (\'ativo_perene\', \'ativo_em_lancamento\', \'desativado\')\n      AND LOWER(sf.product_name) NOT LIKE \'%ziraldo%\'\n      AND LOWER(sf.product_name) NOT LIKE \'% xp%\'\n      AND LOWER(sf.product_name) NOT LIKE \'%maluquinho%\'\n      AND LOWER(sf.product_name) NOT LIKE \'% b2b %\'\n    GROUP BY sf.product_name, ss.product_status, article_name, sf.article_unit\n),\n\nprincipal AS (\n    SELECT\n        product_name,\n        product_status,\n        article_name AS tecido_principal,\n        article_unit,\n        ROUND(CAST(median_article_consumption AS FLOAT64), 4) AS consumo_mediano,\n        minimum_volume_per_order,\n        custo_unitario_min,\n        custo_unitario_max,\n        n_malharias,\n        malharias_nomes,\n        COUNT(*) OVER (PARTITION BY product_name) AS qtd_tecidos_total\n    FROM product_article_agg\n    QUALIFY ROW_NUMBER() OVER (\n        PARTITION BY product_name\n        ORDER BY median_article_consumption DESC\n    ) = 1\n)\n\nSELECT\n    p.product_name,\n    p.product_status,\n    p.tecido_principal,\n    p.article_unit,\n    p.consumo_mediano,\n    p.minimum_volume_per_order,\n    p.qtd_tecidos_total,\n\n    -- Colunas de governança acrescentadas à query original\n    CAST(p.custo_unitario_min AS FLOAT64) AS custo_unitario_min,\n    CAST(p.custo_unitario_max AS FLOAT64) AS custo_unitario_max,\n    ROUND(CAST(p.custo_unitario_min AS FLOAT64) * p.consumo_mediano, 4) AS custo_mp_min_por_peca,\n    ROUND(CAST(p.custo_unitario_max AS FLOAT64) * p.consumo_mediano, 4) AS custo_mp_max_por_peca,\n    p.n_malharias,\n    p.malharias_nomes,\n\n    ltc.lt_malharia_cadastrado_min_dias,\n    ltc.lt_malharia_cadastrado_max_dias,\n    ltr.lt_malharia_realizado_mediano_dias,\n    ltr.n_pedidos_mp_faturados,\n    ltr.n_pedidos_mp_atrasados,\n    SAFE_DIVIDE(ltr.n_pedidos_mp_atrasados, ltr.n_pedidos_mp_faturados) AS pct_pedidos_mp_atrasados,\n\n    ec.cmv_unitario_medio,\n    ec.cmv_unitario_medio_sem_imposto,\n    ec.cmv_data_ref,\n    ec.markup_price_medio,\n    SAFE_DIVIDE(ec.markup_price_medio, ec.cmv_unitario_medio_sem_imposto) AS markup_ratio_medio,\n    SAFE_DIVIDE(CAST(p.custo_unitario_min AS FLOAT64) * p.consumo_mediano, ec.cmv_unitario_medio) AS pct_mp_no_cmv,\n\n    -- GAP DECLARADO: sem fonte no data lake.\n    CAST(NULL AS FLOAT64) AS gramatura_g_m2,\n    CAST(NULL AS FLOAT64) AS largura_cm\nFROM principal AS p\nLEFT JOIN lt_cadastrado_por_artigo AS ltc ON ltc.article_name_norm = p.tecido_principal\nLEFT JOIN lt_realizado_por_artigo  AS ltr ON ltr.article_name_norm = p.tecido_principal\nLEFT JOIN economics_por_produto    AS ec  ON ec.product_name = p.product_name\nORDER BY p.product_name ASC',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
mart_produto_mp

## 6. Contrato de schema

**É o bloco mais importante do notebook.** A base tem consumidor externo (Sheets → Lovable), e o
que quebra uma integração não é o dado errado — é a coluna que mudou de nome, de posição ou de
tipo sem ninguém avisar.

`SCHEMA_CONTRATO` declara, para cada tabela: coluna, tipo e descrição, **na ordem oficial**.
`aplicar_contrato()` valida cada DataFrame contra ele e **levanta erro** se faltar coluna, sobrar
coluna ou o tipo não converter. Também recusa qualquer célula `list`/`dict` — nem Sheets nem
Lovable ingerem estrutura aninhada.

O `dicionario_dados` é gerado do **mesmo dicionário**, de propósito: contrato e documentação não
podem ser duas fontes de verdade que divergem com o tempo.


In [ ]:
# =============================================================================================
# CONTRATO DE SCHEMA — fonte única de verdade sobre colunas, tipos, ordem e significado.
# Formato: (coluna, tipo, descrição). Tipos: STRING | INT64 | FLOAT64 | DATE | BOOL.
# =============================================================================================
SCHEMA_CONTRATO = {
    "dim_mp_fornecimento": {
        "grao": "fabric_sku_id",
        "descricao": "Cotação de tecido numa malharia. Inclui cotação desativada como histórico.",
        "colunas": [
            ("fabric_sku_id", "INT64", "PK. Id da cotação do tecido na malharia."),
            ("fabric_sku", "STRING", "Código do fabric SKU."),
            ("fabric_id", "INT64", "FK do tecido. Conecta com fact_sku_bom."),
            ("fabric_name", "STRING", "Nome do tecido."),
            ("article_id", "INT64", "FK do artigo (MP genérica)."),
            ("article_name", "STRING", "Nome do artigo: a matéria-prima."),
            ("article_unit", "STRING", "Unidade de medida do artigo (kg, metro)."),
            ("knitting_factory_id", "INT64", "FK da malharia."),
            ("malharia_supplier_id", "INT64", "FK do fornecedor. SEMPRE papel de malharia."),
            ("malharia_nome", "STRING", "Nome curto da malharia."),
            ("malharia_razao_social", "STRING", "Razão social da malharia."),
            ("malharia_cnpj", "STRING", "CNPJ da malharia."),
            ("malharia_tipo", "STRING", "Papel do fornecedor. Deve ser sempre 'knitting'."),
            ("malharia_cidade", "STRING", "Cidade da malharia."),
            ("malharia_uf", "STRING", "UF da malharia."),
            ("tipo_frete", "STRING", "Tipo de frete. Frete pode não estar em custo_unitario."),
            ("status_cotacao", "STRING", "Status da cotação. 'available' = vigente."),
            ("custo_unitario", "FLOAT64", "Preço unitário cotado nesta malharia."),
            ("volume_minimo_pedido", "INT64", "Volume mínimo de pedido do fabric SKU."),
            ("multiplo_volume_pedido", "INT64", "Múltiplo de lote de pedido."),
            ("volume_por_rolo", "INT64", "Volume por rolo."),
            ("codigo_cor_fabrica", "STRING", "Código de cor usado pela fábrica."),
            ("codigo_tecido_nf", "STRING", "Código do tecido na nota fiscal."),
            ("nome_tecido_nf", "STRING", "Nome do tecido na nota fiscal."),
            ("payment_agreement_id", "INT64", "FK do acordo de pagamento da cotação."),
            ("acordo_pagamento_label", "STRING", "Rótulo do acordo de pagamento."),
            ("forma_pagamento", "STRING", "Forma de pagamento acordada."),
            ("parcelas", "INT64", "Número de parcelas."),
            ("dias_primeiro_vencimento", "INT64", "Dias até o primeiro vencimento."),
            ("status_acordo_fornecedor", "STRING", "Status do acordo vigente do fornecedor."),
            ("lt_tingimento_cadastrado_dias", "INT64", "LT de tingimento cadastrado."),
            ("lt_producao_cadastrado_dias", "INT64", "LT de produção cadastrado."),
            ("lt_malharia_cadastrado_dias", "INT64", "LT MALHARIA CADASTRADO = tingimento + produção. NULL = sem cadastro."),
            ("volume_minimo_pedido_artigo", "INT64", "Volume mínimo de pedido no nível do artigo."),
            ("multiplo_volume_tingimento", "INT64", "Múltiplo de volume para tingimento."),
            ("comprado_pelo_fornecedor", "BOOL", "Se a MP é comprada pelo próprio fornecedor."),
            ("gramatura_g_m2", "FLOAT64", "GAP: sem fonte no data lake. Sempre nulo hoje."),
            ("largura_cm", "FLOAT64", "GAP: sem fonte no data lake. Sempre nulo hoje."),
            ("product_color_id", "INT64", "Cor de produto à qual o tecido está amarrado."),
            ("fabric_amarrado_a_cor", "BOOL", "Tecido amarrado a cor. Hoje TRUE em 100% dos casos."),
            ("malharia_field_errors", "STRING", "JSON de erros de cadastro do fornecedor. '{}' = OK."),
            ("malharia_cadastro_incompleto", "BOOL", "Derivado: cadastro do fornecedor tem erro."),
            ("origem_ultima_alteracao", "STRING", "Origem da última alteração (manual vs sistema)."),
            ("autor_ultima_alteracao", "STRING", "Autor da última alteração."),
            ("cotacao_criada_em", "DATE", "Data de criação da cotação."),
            ("cotacao_atualizada_em", "DATE", "Data da última atualização da cotação."),
            ("custo_min_fabric", "FLOAT64", "Menor custo vigente do tecido entre malharias."),
            ("custo_max_fabric", "FLOAT64", "Maior custo vigente do tecido entre malharias."),
            ("n_malharias", "INT64", "Malharias que fornecem o tecido. 1 = fonte única."),
            ("malharias_nomes", "STRING", "Malharias vigentes, separadas por ';'. Nunca ARRAY."),
        ],
    },
    "fact_sku_bom": {
        "grao": "sku x fabric_id",
        "descricao": "Ficha técnica real: um tecido do SKU por linha.",
        "colunas": [
            ("sku", "STRING", "PK parte 1. Código do SKU."),
            ("sku_name", "STRING", "Nome do SKU."),
            ("product_sku_id", "INT64", "Id interno do SKU no Muninn."),
            ("product_id", "INT64", "FK do produto."),
            ("product_name", "STRING", "Nome do produto."),
            ("sku_state", "STRING", "Estado comercial do SKU."),
            ("gender", "STRING", "Gênero."),
            ("color", "STRING", "Cor."),
            ("size", "STRING", "Tamanho."),
            ("category", "STRING", "Categoria."),
            ("family", "STRING", "Família."),
            ("product_state_id", "INT64", "Estado do ciclo de vida do SKU no Muninn."),
            ("fabric_id", "INT64", "PK parte 2. FK do tecido. Conecta com dim_mp_fornecimento."),
            ("fabric_name", "STRING", "Nome do tecido."),
            ("article_id", "INT64", "FK do artigo."),
            ("article_name", "STRING", "Nome do artigo: a matéria-prima."),
            ("article_unit", "STRING", "Unidade de medida do artigo."),
            ("consumo_sku", "FLOAT64", "Consumo do tecido por peça. NUNCA somar com consumo de grão produto."),
            ("custo_unitario_min_fabric", "FLOAT64", "Menor custo unitário vigente do tecido."),
            ("custo_unitario_max_fabric", "FLOAT64", "Maior custo unitário vigente do tecido."),
            ("n_malharias", "INT64", "Malharias que fornecem o tecido."),
            ("malharias_nomes", "STRING", "Malharias vigentes, separadas por ';'."),
            ("custo_mp_min", "FLOAT64", "Custo mínimo de MP deste tecido na peça = consumo x custo."),
            ("custo_mp_max", "FLOAT64", "Custo máximo de MP deste tecido na peça."),
            ("custo_mp_referencia_skus", "FLOAT64", "Custo de MP já calculado em integrated.skus. Reconciliação."),
            ("presente_em_sku_bom", "BOOL", "Chave presente em sku_bill_of_materials. Reconciliação."),
        ],
    },
    "fact_mp_lt_realizado": {
        "grao": "fabric_order_id",
        "descricao": "Lead time realizado da malharia, por pedido de tecido.",
        "colunas": [
            ("fabric_order_id", "INT64", "PK. Id do pedido de tecido."),
            ("fabric_order_code", "STRING", "Código do pedido de tecido."),
            ("fabric_sku_id", "INT64", "FK da cotação. Conecta com dim_mp_fornecimento."),
            ("fabric_sku", "STRING", "Código do fabric SKU."),
            ("fabric_id", "INT64", "FK do tecido."),
            ("fabric_name", "STRING", "Nome do tecido."),
            ("article_id", "INT64", "FK do artigo."),
            ("article_name", "STRING", "Nome do artigo: a matéria-prima."),
            ("article_unit", "STRING", "Unidade de medida do artigo."),
            ("knitting_factory_id", "INT64", "FK da malharia."),
            ("malharia_nome", "STRING", "Nome curto da malharia."),
            ("malharia_razao_social", "STRING", "Razão social da malharia."),
            ("status_pedido", "STRING", "Status do pedido de tecido."),
            ("volume_pedido", "FLOAT64", "Volume pedido."),
            ("volume_faturado", "FLOAT64", "Volume efetivamente faturado."),
            ("volume_disponivel", "FLOAT64", "Volume disponível."),
            ("preco_pedido", "FLOAT64", "Preço no momento do pedido."),
            ("preco_faturado", "FLOAT64", "Preço efetivamente faturado."),
            ("data_pedido", "DATE", "Data de criação do pedido. Início do lead time."),
            ("data_faturamento_estimada_original", "DATE", "Compromisso ORIGINAL. Base da métrica de contrato."),
            ("data_faturamento_repactuada", "DATE", "Compromisso REPACTUADO. Só diagnóstico."),
            ("data_faturamento_real", "DATE", "Faturamento real. Fim do lead time."),
            ("lt_malharia_realizado_dias", "INT64", "LT MALHARIA REALIZADO: pedido -> faturamento real."),
            ("lt_malharia_estimado_original_dias", "INT64", "LT prometido originalmente, no mesmo eixo."),
            ("atraso_vs_original_dias", "INT64", "METRICA DE CONTRATO. Atraso vs compromisso original."),
            ("atraso_vs_repactuado_dias", "INT64", "Diagnóstico. Atraso vs compromisso repactuado."),
            ("dias_repactuacao", "INT64", "Diagnóstico. Quanto a data foi empurrada."),
            ("lt_malharia_cadastrado_dias", "INT64", "LT cadastrado no mesmo grão. NULL = sem cadastro."),
            ("desvio_vs_cadastrado_dias", "INT64", "Realizado menos cadastrado. Só válido onde há cadastro."),
            ("status_faturamento_mp", "STRING", "MISSING_ESTIMATED | PENDENTE | LATE | ON_TIME | EARLY."),
        ],
    },
    "fact_sku_economics": {
        "grao": "sku",
        "descricao": "CMV, markup e custo de MP por SKU. ATENCAO: escopos de custo diferentes.",
        "colunas": [
            ("sku", "STRING", "PK. Código do SKU."),
            ("sku_name", "STRING", "Nome do SKU."),
            ("product_id", "INT64", "FK do produto."),
            ("product_name", "STRING", "Nome do produto."),
            ("sku_state", "STRING", "Estado comercial do SKU."),
            ("gender", "STRING", "Gênero."),
            ("color", "STRING", "Cor."),
            ("size", "STRING", "Tamanho."),
            ("category", "STRING", "Categoria."),
            ("family", "STRING", "Família."),
            ("n_tecidos_no_sku", "INT64", "Quantidade de tecidos na ficha técnica do SKU."),
            ("consumo_total_sku", "FLOAT64", "Soma do consumo de todos os tecidos do SKU."),
            ("custo_mp_min", "FLOAT64", "ESCOPO MP ISOLADA. Custo mínimo de matéria-prima da peça."),
            ("custo_mp_max", "FLOAT64", "ESCOPO MP ISOLADA. Custo máximo de matéria-prima da peça."),
            ("custo_mp_referencia_skus", "FLOAT64", "Custo de MP já calculado em integrated.skus."),
            ("cmv_data_ref", "DATE", "Data do CMV usado. Mede o frescor do dado."),
            ("cmv_unitario", "FLOAT64", "ESCOPO CUSTO TOTAL. Não comparar direto com custo_mp."),
            ("cmv_unitario_sem_imposto", "FLOAT64", "CMV unitário livre de impostos."),
            ("cmv_qtd_vendida_ref", "FLOAT64", "Quantidade vendida na data de referência do CMV."),
            ("markup_price", "FLOAT64", "Preço de markup vigente."),
            ("markup_valid_from", "DATE", "Início de vigência do preço de markup."),
            ("sku_price", "FLOAT64", "Preço do SKU."),
            ("full_price", "FLOAT64", "Preço cheio."),
            ("markup_ratio", "FLOAT64", "markup_price / cmv_unitario_sem_imposto."),
            ("pct_mp_no_cmv", "FLOAT64", "custo_mp_min / cmv_unitario. Fora de (0,1] = escopo trocado."),
            ("custo_variante_unitario", "FLOAT64", "Custo unitário da variante (product_cost). Escopo distinto."),
            ("custo_sku_muninn", "FLOAT64", "sku_cost do Muninn. Escopo distinto."),
            ("custo_industrializacao", "FLOAT64", "Custo de industrialização. Escopo distinto."),
            ("custo_sku_industrializado", "FLOAT64", "Custo do SKU industrializado. Escopo distinto."),
            ("custo_produto_acabado_fornecedor", "FLOAT64", "Custo de produto acabado do fornecedor."),
        ],
    },
    "mart_produto_mp": {
        "grao": "product_name",
        "descricao": "Camada comercial derivada: venda L8M, tecido principal por produto.",
        "colunas": [
            ("product_name", "STRING", "PK. Nome do produto."),
            ("product_status", "STRING", "Status do produto por prioridade de sku_state."),
            ("tecido_principal", "STRING", "Artigo de maior consumo mediano. Nome normalizado."),
            ("article_unit", "STRING", "Unidade de medida do artigo."),
            ("consumo_mediano", "FLOAT64", "Consumo mediano do tecido principal por peça."),
            ("minimum_volume_per_order", "INT64", "Volume mínimo de pedido."),
            ("qtd_tecidos_total", "INT64", "Quantidade de artigos distintos no produto."),
            ("custo_unitario_min", "FLOAT64", "Menor custo unitário do tecido principal."),
            ("custo_unitario_max", "FLOAT64", "Maior custo unitário do tecido principal."),
            ("custo_mp_min_por_peca", "FLOAT64", "custo_unitario_min x consumo_mediano."),
            ("custo_mp_max_por_peca", "FLOAT64", "custo_unitario_max x consumo_mediano."),
            ("n_malharias", "INT64", "Malharias que fornecem o tecido principal."),
            ("malharias_nomes", "STRING", "Malharias, separadas por ';'."),
            ("lt_malharia_cadastrado_min_dias", "INT64", "Menor LT cadastrado do artigo."),
            ("lt_malharia_cadastrado_max_dias", "INT64", "Maior LT cadastrado do artigo."),
            ("lt_malharia_realizado_mediano_dias", "INT64", "LT realizado mediano do artigo."),
            ("n_pedidos_mp_faturados", "INT64", "Pedidos de MP faturados. Base do LT realizado."),
            ("n_pedidos_mp_atrasados", "INT64", "Pedidos faturados após o compromisso original."),
            ("pct_pedidos_mp_atrasados", "FLOAT64", "Proporção de pedidos de MP atrasados."),
            ("cmv_unitario_medio", "FLOAT64", "CMV unitário médio dos SKUs do produto."),
            ("cmv_unitario_medio_sem_imposto", "FLOAT64", "CMV médio livre de impostos."),
            ("cmv_data_ref", "DATE", "Data mais recente de CMV no produto."),
            ("markup_price_medio", "FLOAT64", "Preço de markup médio dos SKUs do produto."),
            ("markup_ratio_medio", "FLOAT64", "markup_price_medio / cmv medio sem imposto."),
            ("pct_mp_no_cmv", "FLOAT64", "Peso da MP no CMV do produto."),
            ("gramatura_g_m2", "FLOAT64", "GAP: sem fonte no data lake. Sempre nulo hoje."),
            ("largura_cm", "FLOAT64", "GAP: sem fonte no data lake. Sempre nulo hoje."),
        ],
    },
}

TIPOS_VALIDOS = {"STRING", "INT64", "FLOAT64", "DATE", "BOOL"}


def _coagir(serie: pd.Series, tipo: str) -> pd.Series:
    """Converte a coluna para o tipo do contrato, preservando NULL como ausência real."""
    if tipo == "STRING":
        return serie.astype("string")
    if tipo == "INT64":
        return pd.to_numeric(serie, errors="coerce").round().astype("Int64")
    if tipo == "FLOAT64":
        return pd.to_numeric(serie, errors="coerce").astype("float64")
    if tipo == "BOOL":
        return serie.astype("boolean")
    if tipo == "DATE":
        # ISO YYYY-MM-DD como texto: é o formato que Sheets e Lovable ingerem sem ambiguidade.
        return pd.to_datetime(serie, errors="coerce").dt.strftime("%Y-%m-%d").astype("string")
    raise ValueError(f"Tipo fora do contrato: {tipo}")


def aplicar_contrato(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """Valida e normaliza um DataFrame contra SCHEMA_CONTRATO. Levanta erro em qualquer desvio."""
    if nome not in SCHEMA_CONTRATO:
        raise KeyError(f"'{nome}' não está declarado em SCHEMA_CONTRATO")

    spec = SCHEMA_CONTRATO[nome]["colunas"]
    esperadas = [c for c, _, _ in spec]

    faltando = [c for c in esperadas if c not in df.columns]
    sobrando = [c for c in df.columns if c not in esperadas]
    if faltando or sobrando:
        raise ValueError(
            f"[{nome}] contrato violado.\n"
            f"  Colunas faltando : {faltando}\n"
            f"  Colunas sobrando : {sobrando}"
        )

    out = pd.DataFrame(index=df.index)
    for coluna, tipo, _ in spec:
        if tipo not in TIPOS_VALIDOS:
            raise ValueError(f"[{nome}.{coluna}] tipo inválido no contrato: {tipo}")
        out[coluna] = _coagir(df[coluna], tipo)

    # Sheets e Lovable não ingerem estrutura aninhada. Falha alto em vez de exportar lixo.
    for coluna in out.columns:
        amostra = out[coluna].dropna().head(200)
        if any(isinstance(v, (list, dict, set, tuple, np.ndarray)) for v in amostra):
            raise TypeError(f"[{nome}.{coluna}] contém valor aninhado — saída precisa ser plana")

    out["atualizado_em"] = ATUALIZADO_EM
    return out.reset_index(drop=True)


# --- Aplica o contrato a todas as tabelas -----------------------------------------------------
TABELAS_BRUTAS = {
    "dim_mp_fornecimento": dim_mp_fornecimento,
    "fact_sku_bom": fact_sku_bom,
    "fact_mp_lt_realizado": fact_mp_lt_realizado,
    "fact_sku_economics": fact_sku_economics,
    "mart_produto_mp": mart_produto_mp,
}

TABELAS = {nome: aplicar_contrato(df, nome) for nome, df in TABELAS_BRUTAS.items()}

dim_mp_fornecimento_final = TABELAS["dim_mp_fornecimento"]
fact_sku_bom_final = TABELAS["fact_sku_bom"]
fact_mp_lt_realizado_final = TABELAS["fact_mp_lt_realizado"]
fact_sku_economics_final = TABELAS["fact_sku_economics"]
mart_produto_mp_final = TABELAS["mart_produto_mp"]

# --- Dicionário de dados, gerado do MESMO dicionário do contrato ------------------------------
# Contrato e documentação não podem ser duas fontes de verdade que divergem com o tempo.
dicionario_dados = pd.DataFrame(
    [
        {
            "tabela": tabela,
            "grao": spec["grao"],
            "descricao_tabela": spec["descricao"],
            "posicao": i + 1,
            "coluna": coluna,
            "tipo": tipo,
            "descricao_coluna": descricao,
        }
        for tabela, spec in SCHEMA_CONTRATO.items()
        for i, (coluna, tipo, descricao) in enumerate(spec["colunas"])
    ]
)

print("Contrato aplicado com sucesso.\n")
for nome, df in TABELAS.items():
    print(f"  {nome:<24} {len(df):>7,} linhas  x  {df.shape[1]:>3} colunas")
print(f"\n  {'dicionario_dados':<24} {len(dicionario_dados):>7,} linhas")


## 7. Auditoria de qualidade

Gera `df_governanca`: uma linha por check, com valor, baseline e status. É tabela plana e
também é exportada — o consumidor consegue ver a saúde do cadastro sem abrir o notebook.

Três níveis de severidade:

- **FALHA** — quebra o contrato ou a integridade do grão. Para a esteira.
- **ATENCAO** — degradou em relação ao baseline de 2026-08-07, ou valor fora da faixa esperada.
- **OK** / **GAP_CONHECIDO** — dentro do esperado. `GAP_CONHECIDO` é lacuna já declarada
  (gramatura, largura), não regressão.

O check de reconciliação é o mais importante: hoje `sku_bill_of_materials` e
`muninn_product_skus_fabrics` concordam em 100% do consumo. **Qualquer linha divergente vira
FALHA** — é o sinal de que as duas fontes começaram a se separar em silêncio.


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

recon_bom = _dntk.execute_sql(
  '-- Reconciliação entre as duas fontes de ficha técnica.\n-- Baseline 2026-08-07: 24.388 casando, 0 divergentes, 11.751 só no sku_bill_of_materials.\n-- Qualquer linha em n_consumo_divergente é FALHA: significa que as fontes se separaram.\nWITH bom AS (\n    SELECT sku, fabric_id, consumption\n    FROM `insider-data-lake.integrated.sku_bill_of_materials`\n),\npsf AS (\n    SELECT mps.sku, f.fabric_id, f.consumption\n    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS f\n    JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps USING (product_sku_id)\n)\nSELECT\n    (SELECT COUNT(*) FROM psf)                                                     AS n_psf,\n    (SELECT COUNT(*) FROM bom)                                                     AS n_bom_derivado,\n    (SELECT COUNT(*) FROM bom JOIN psf USING (sku, fabric_id)\n      WHERE bom.consumption = psf.consumption)                                     AS n_consumo_igual,\n    (SELECT COUNT(*) FROM bom JOIN psf USING (sku, fabric_id)\n      WHERE bom.consumption != psf.consumption)                                    AS n_consumo_divergente,\n    (SELECT COUNT(*) FROM bom LEFT JOIN psf USING (sku, fabric_id)\n      WHERE psf.sku IS NULL)                                                       AS n_so_no_bom_derivado,\n    (SELECT COUNT(*) FROM `insider-data-lake.integrated.muninn_fabrics`)           AS n_fabrics,\n    (SELECT COUNTIF(product_color_id IS NOT NULL)\n       FROM `insider-data-lake.integrated.muninn_fabrics`)                         AS n_fabrics_com_cor,\n    (SELECT COUNT(*) FROM `insider-data-lake.integrated.muninn_articles`)          AS n_artigos,\n    (SELECT COUNT(DISTINCT article_id)\n       FROM `insider-data-lake.integrated.muninn_articles_knitting_factories`)     AS n_artigos_com_lt_cadastrado,\n    (SELECT MAX(date) FROM `insider-data-lake.integrated.cmv_model`)               AS cmv_data_mais_recente',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
recon_bom

In [ ]:
checks = []


def check(nome, valor, status, detalhe, baseline=None):
    checks.append(
        {
            "check": nome,
            "valor": valor,
            "baseline_20260807": baseline,
            "status": status,
            "detalhe": detalhe,
        }
    )


def status_pk(df, colunas):
    """FALHA se a chave não for única — PK duplicada quebra modelagem relacional no destino."""
    dup = int(df.duplicated(subset=colunas).sum())
    return dup, ("OK" if dup == 0 else "FALHA")


r = recon_bom.iloc[0]

# --- 1. Unicidade de grão ---------------------------------------------------------------------
for nome, colunas in [
    ("dim_mp_fornecimento", ["fabric_sku_id"]),
    ("fact_sku_bom", ["sku", "fabric_id"]),
    ("fact_mp_lt_realizado", ["fabric_order_id"]),
    ("fact_sku_economics", ["sku"]),
    ("mart_produto_mp", ["product_name"]),
]:
    dup, st = status_pk(TABELAS[nome], colunas)
    check(f"pk_unica__{nome}", dup, st, f"Linhas duplicadas em ({', '.join(colunas)}). 0 = OK.", 0)

# --- 2. Volume das tabelas --------------------------------------------------------------------
check("linhas__dim_mp_fornecimento", len(TABELAS["dim_mp_fornecimento"]), "OK",
      "Cotações de tecido cadastradas.", BASELINE["dim_mp_fornecimento_linhas"])
check("linhas__fact_sku_bom", len(TABELAS["fact_sku_bom"]), "OK",
      "Linhas de ficha técnica (sku x tecido).", BASELINE["fact_sku_bom_linhas"])
check("linhas__mart_produto_mp", len(TABELAS["mart_produto_mp"]), "OK",
      "Produtos no recorte comercial. Varia com a janela de venda L8M.", BASELINE["mart_produtos"])

# --- 3. Reconciliação entre fontes de ficha técnica (check crítico) ---------------------------
divergentes = int(r["n_consumo_divergente"])
check("reconciliacao__consumo_divergente", divergentes,
      "OK" if divergentes == 0 else "FALHA",
      "Chaves (sku, fabric_id) com consumo diferente entre sku_bill_of_materials e "
      "muninn_product_skus_fabrics. Qualquer valor > 0 significa que as fontes se separaram.",
      BASELINE["reconciliacao_consumo_divergente"])
check("reconciliacao__so_no_bom_derivado", int(r["n_so_no_bom_derivado"]), "OK",
      "Linhas presentes só em sku_bill_of_materials. Esperado: é superset.",
      BASELINE["sku_bom_derivado_linhas"] - BASELINE["fact_sku_bom_linhas"])

# --- 4. Papel de fornecedor -------------------------------------------------------------------
# Erro mais fácil de cometer: puxar o fornecedor de confecção em vez da malharia.
tipos = set(TABELAS["dim_mp_fornecimento"]["malharia_tipo"].dropna().unique())
check("papel_fornecedor__somente_knitting", "; ".join(sorted(tipos)) or "(vazio)",
      "OK" if tipos == {"knitting"} else "FALHA",
      "Papéis de fornecedor presentes. Só 'knitting' é válido — 'manufacturer' indica que o "
      "join pegou o fornecedor de confecção.", "knitting")

# --- 5. Cobertura de cadastro -----------------------------------------------------------------
dim = TABELAS["dim_mp_fornecimento"]

for coluna in ("gramatura_g_m2", "largura_cm"):
    preenchidos = int(dim[coluna].notna().sum())
    check(f"cobertura__{coluna}", preenchidos,
          "GAP_CONHECIDO" if preenchidos == 0 else "OK",
          "Sem fonte no data lake. 0 preenchidos é a lacuna já declarada, não regressão.", 0)

sem_lt = int(dim["lt_malharia_cadastrado_dias"].isna().sum())
pct_sem_lt = round(100 * sem_lt / max(len(dim), 1), 1)
check("cobertura__sem_lt_malharia_cadastrado", f"{sem_lt} ({pct_sem_lt}%)",
      "OK" if pct_sem_lt <= 20 else "ATENCAO",
      "Cotações sem LT de malharia cadastrado. Comparação cadastrado vs realizado só vale no "
      "subconjunto coberto.", None)

check("cobertura__artigos_com_lt_cadastrado",
      f"{int(r['n_artigos_com_lt_cadastrado'])} de {int(r['n_artigos'])}", "OK",
      "Artigos com LT de malharia cadastrado.", None)

pct_cor = round(100 * int(r["n_fabrics_com_cor"]) / max(int(r["n_fabrics"]), 1), 1)
check("semantica__fabric_amarrado_a_cor", f"{pct_cor}%", "OK",
      "Tecidos amarrados a uma cor de produto. 100% confirma que tecido NÃO é entidade "
      "genérica — falta a dimensão de cor na hierarquia documentada.", "100%")

incompletos = int(dim["malharia_cadastro_incompleto"].fillna(False).sum())
check("qualidade__malharia_cadastro_incompleto", incompletos,
      "OK" if incompletos == 0 else "ATENCAO",
      "Cotações cuja malharia tem erro de cadastro (field_errors preenchido).", None)

# --- 6. Frescor do CMV ------------------------------------------------------------------------
atraso_cmv = (pd.Timestamp(DATA_REFERENCIA) - pd.Timestamp(r["cmv_data_mais_recente"])).days
check("frescor__cmv_dias_atraso", atraso_cmv,
      "OK" if atraso_cmv <= 3 else "ATENCAO",
      f"Dias entre hoje e a data mais recente de cmv_model ({r['cmv_data_mais_recente']}).", None)

eco = TABELAS["fact_sku_economics"]
com_cmv = int(eco["cmv_unitario"].notna().sum())
check("cobertura__skus_com_cmv", f"{com_cmv} de {len(eco)}", "OK",
      "SKUs da base de MP que têm CMV calculado.", None)
check("cobertura__skus_com_markup", f"{int(eco['markup_price'].notna().sum())} de {len(eco)}",
      "OK", "SKUs da base de MP com preço de markup vigente.", None)

# --- 7. Sanidade de valor ---------------------------------------------------------------------
custo_invalido = int((dim["custo_unitario"].fillna(0) <= 0).sum())
check("sanidade__custo_unitario_invalido", custo_invalido,
      "OK" if custo_invalido == 0 else "ATENCAO",
      "Cotações com custo unitário nulo ou <= 0.", 0)

bom = TABELAS["fact_sku_bom"]
consumo_invalido = int((bom["consumo_sku"].fillna(0) <= 0).sum())
check("sanidade__consumo_sku_invalido", consumo_invalido,
      "OK" if consumo_invalido == 0 else "ATENCAO",
      "Linhas de ficha técnica com consumo nulo ou <= 0.", 0)

lt = TABELAS["fact_mp_lt_realizado"]
lt_negativo = int((lt["lt_malharia_realizado_dias"] < 0).sum())
check("sanidade__lt_realizado_negativo", lt_negativo,
      "OK" if lt_negativo == 0 else "ATENCAO",
      "Pedidos com faturamento real anterior à criação. Inconsistência de cadastro — listada, "
      "nunca silenciada.", 0)

pct = eco["pct_mp_no_cmv"].dropna()
fora_faixa = int(((pct <= 0) | (pct > 1)).sum())
check("sanidade__pct_mp_no_cmv_fora_faixa", fora_faixa,
      "OK" if fora_faixa == 0 else "ATENCAO",
      "SKUs com participação de MP fora de (0, 1]. Indica escopo de custo trocado — MP isolada "
      "não pode superar o custo total.", 0)

# --- 8. Aderência de lead time da malharia ----------------------------------------------------
faturados = lt[lt["status_faturamento_mp"].isin(["LATE", "ON_TIME", "EARLY"])]
if len(faturados):
    pct_late = round(100 * (faturados["status_faturamento_mp"] == "LATE").mean(), 1)
    check("lead_time__pct_pedidos_mp_atrasados", f"{pct_late}%", "OK",
          f"Pedidos faturados após o compromisso ORIGINAL, sobre {len(faturados)} faturados. "
          "Métrica de contrato.", None)
    repactuados = int((lt["dias_repactuacao"].fillna(0) > 0).sum())
    check("lead_time__pedidos_com_repactuacao", repactuados, "OK",
          "Pedidos cuja data de faturamento foi empurrada. Se for alto, medir só contra a data "
          "repactuada esconderia a maior parte do atraso.", None)

# --- Consolidação -----------------------------------------------------------------------------
df_governanca = pd.DataFrame(checks)
df_governanca["atualizado_em"] = ATUALIZADO_EM

falhas = df_governanca[df_governanca["status"] == "FALHA"]
atencoes = df_governanca[df_governanca["status"] == "ATENCAO"]

print(f"Checks: {len(df_governanca)}  |  FALHA: {len(falhas)}  |  ATENCAO: {len(atencoes)}\n")
for _, linha in df_governanca.iterrows():
    print(f"  [{linha['status']:<14}] {linha['check']:<42} {linha['valor']}")

if len(falhas):
    print("\nFALHAS:")
    for _, linha in falhas.iterrows():
        print(f"  - {linha['check']}: {linha['detalhe']}")
    raise AssertionError(
        f"{len(falhas)} check(s) de governança falharam — a base não deve ser publicada assim."
    )


## 8. Exportação

Ponto **único** de saída. A camada de extração para Google Sheets pluga em `escrever_sheets()`
sem tocar em nenhuma lógica acima — é por isso que ela existe já como stub, controlada por
`ESCREVER_SHEETS` (hoje `False`, não escreve nada).

Todas as tabelas saem em `exports/` como `governanca_mp_<tabela>.csv`, com codificação UTF-8 e
o schema garantido pelo contrato do bloco 6.


In [ ]:
"""Camada de publicação governada para Google Sheets.

O notebook Deepnote mantém uma cópia desta implementação no bloco de exportação,
pois o runtime hospedado não monta este diretório local.
"""

from __future__ import annotations

import importlib.util
import json
import os
from datetime import date, datetime
from decimal import Decimal
from typing import Any

import numpy as np
import pandas as pd

SHEETS_URL = "https://docs.google.com/spreadsheets/d/1XB9cZztzBziergCs7AzaeUMUzlM-Tg9vooakHSqaamk/edit"
SHEETS_BATCH_SIZE = 2_000
SHEETS_DESTINOS = (
    ("dim_mp_fornecimento", 0, "dim_mp_fornecimento"),
    ("fact_sku_bom", 1263944509, "fact_sku_bom"),
    ("fact_mp_lt_realizado", 784021430, "fact_mp_leadtime_realizado"),
    ("fact_sku_economics", 1858110063, "fact_sku_economics"),
    ("mart_produto_mp", 2006224731, "mart_produto_mp"),
    ("dicionario_dados", 1605547778, "dicionario_dados"),
    ("governanca", 1425809795, "df_governanca"),
)


def ensure_sheets_dependencies() -> None:
    """Install gspread only when the ephemeral Deepnote runtime needs it."""
    missing = [
        package
        for package, module in (("gspread", "gspread"), ("gspread-dataframe", "gspread_dataframe"))
        if importlib.util.find_spec(module) is None
    ]
    if missing:
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


def get_gspread_client():
    """Authenticate without logging credential contents, preferring Deepnote's service account."""
    ensure_sheets_dependencies()

    import google.auth
    import google.auth.transport.requests
    import google.oauth2.service_account
    import gspread

    scopes = (
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    )
    for variable in (
        "GOOGLE_SERVICE_ACCOUNT_JSON",
        "BIGQUERY_INTEGRATION_SERVICE_ACCOUNT",
        "BQ_SERVICE_ACCOUNT",
    ):
        raw = os.getenv(variable)
        if raw:
            credentials = google.oauth2.service_account.Credentials.from_service_account_info(
                json.loads(raw), scopes=scopes
            )
            print(f"GSheets auth: {variable}")
            return gspread.authorize(credentials)

    credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    if credentials_path and os.path.exists(credentials_path):
        credentials = google.oauth2.service_account.Credentials.from_service_account_file(
            credentials_path, scopes=scopes
        )
        print("GSheets auth: GOOGLE_APPLICATION_CREDENTIALS")
        return gspread.authorize(credentials)

    try:
        credentials, _ = google.auth.default(scopes=scopes)
        if credentials and not credentials.valid:
            credentials.refresh(google.auth.transport.requests.Request())
        print("GSheets auth: Application Default Credentials")
        return gspread.authorize(credentials)
    except Exception as exc:
        raise RuntimeError(
            "Sem credenciais para Google Sheets. Use a service account compartilhada com a planilha."
        ) from exc


def value_for_sheets(value: Any) -> Any:
    """Keep scalars, write dates as ISO, and reject nested data before any clear."""
    if isinstance(value, (list, dict, set, tuple, np.ndarray)):
        raise TypeError(f"Valor aninhado não permitido na exportação: {type(value).__name__}")
    if value is None or value is pd.NA:
        return ""
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, pd.Timestamp):
        return "" if pd.isna(value) else value.date().isoformat()
    if isinstance(value, datetime):
        return value.date().isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, Decimal):
        return float(value)
    if pd.isna(value):
        return ""
    return value


def prepare_for_sheets(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Validate one output and return a scalar-only Sheets payload."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} não é DataFrame.")
    if df.empty:
        raise AssertionError(f"Preflight: {name} está vazio; nenhuma aba será limpa.")
    if df.columns.has_duplicates or any(
        not isinstance(column, str) or not column for column in df.columns
    ):
        raise AssertionError(f"Preflight: {name} tem cabeçalho vazio, inválido ou duplicado.")

    payload = df.copy()
    for column in payload.columns:
        payload[column] = payload[column].map(value_for_sheets)
    return payload


def preflight_outputs(outputs: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Validate the complete publication set before mutating any destination tab."""
    expected = [name for name, _, _ in SHEETS_DESTINOS]
    missing = [name for name in expected if name not in outputs]
    extra = [name for name in outputs if name not in expected]
    if missing or extra:
        raise AssertionError(f"Preflight: SAIDAS divergente. Ausentes={missing}; extras={extra}.")
    return {name: prepare_for_sheets(outputs[name], name) for name in expected}


def resolve_destinations(client):
    """Resolve all worksheet IDs and titles before starting the first overwrite."""
    spreadsheet = client.open_by_url(SHEETS_URL)
    destinations = {}
    for name, gid, expected_title in SHEETS_DESTINOS:
        try:
            worksheet = spreadsheet.get_worksheet_by_id(gid)
        except Exception as exc:
            raise RuntimeError(f"Aba gid={gid} não encontrada para {name}.") from exc
        if worksheet.title != expected_title:
            raise RuntimeError(
                f"Aba gid={gid} mudou de nome: esperado={expected_title!r}; atual={worksheet.title!r}."
            )
        destinations[name] = worksheet
    return destinations


def write_worksheet(worksheet, payload: pd.DataFrame, name: str) -> dict[str, Any]:
    """Overwrite one tab in chunks and prove its structural post-condition."""
    from gspread_dataframe import set_with_dataframe
    from gspread.utils import rowcol_to_a1

    row_count, column_count = payload.shape
    worksheet.clear()
    worksheet.resize(rows=row_count + 1, cols=column_count)
    worksheet.update("A1", [payload.columns.tolist()], value_input_option="RAW")

    for start in range(0, row_count, SHEETS_BATCH_SIZE):
        set_with_dataframe(
            worksheet,
            payload.iloc[start : start + SHEETS_BATCH_SIZE],
            row=start + 2,
            col=1,
            include_index=False,
            include_column_header=False,
            resize=False,
        )

    if worksheet.row_values(1) != payload.columns.tolist():
        raise RuntimeError(f"{name}: cabeçalho pós-escrita divergente.")
    if worksheet.row_count != row_count + 1 or worksheet.col_count != column_count:
        raise RuntimeError(f"{name}: dimensão pós-escrita divergente.")

    last_row = worksheet.get(f"A{row_count + 1}:{rowcol_to_a1(1, column_count)}{row_count + 1}")
    if not last_row or not any(value != "" for value in last_row[0]):
        raise RuntimeError(f"{name}: última linha não foi encontrada após a escrita.")
    return {
        "table": name,
        "worksheet": worksheet.title,
        "gid": worksheet.id,
        "rows": row_count,
        "columns": column_count,
        "status": "success",
    }


def write_outputs_to_sheets(outputs: dict[str, pd.DataFrame]) -> dict[str, Any]:
    """Publish the seven outputs after global data and destination preflight."""
    payloads = preflight_outputs(outputs)
    client = get_gspread_client()
    destinations = resolve_destinations(client)
    results = [
        write_worksheet(destinations[name], payloads[name], name)
        for name, _, _ in SHEETS_DESTINOS
    ]
    return {"written": True, "spreadsheet_url": SHEETS_URL, "tables": results}


# Código complementar do bloco Deepnote; é concatenado após sheets_export.py.
# O runtime hospedado executa a cópia integral das duas fontes no notebook.

SAIDAS = {
    **TABELAS,
    "dicionario_dados": dicionario_dados,
    "governanca": df_governanca,
}


def exportar_csv(df: pd.DataFrame, nome: str) -> str:
    """Grava uma tabela governada em exports/. Ponto único de escrita em disco."""
    os.makedirs(DIR_EXPORT, exist_ok=True)
    caminho = os.path.join(DIR_EXPORT, f"governanca_mp_{nome}.csv")
    df.to_csv(caminho, index=False, encoding="utf-8")
    return caminho


if EXPORTAR_CSV:
    print(f"Exportando {len(SAIDAS)} tabelas para '{DIR_EXPORT}/'\n")
    for nome, df in SAIDAS.items():
        caminho = exportar_csv(df, nome)
        print(f"  {caminho:<50} {len(df):>7,} linhas x {df.shape[1]:>3} colunas")
else:
    print("EXPORTAR_CSV = False — nenhum arquivo escrito.")

if ESCREVER_SHEETS:
    SHEETS_EXPORT_RESULT = write_outputs_to_sheets(SAIDAS)
    for resultado in SHEETS_EXPORT_RESULT["tables"]:
        print(
            f"Sheets: {resultado['table']} → {resultado['worksheet']} | "
            f"{resultado['rows']:,} linhas × {resultado['columns']} colunas"
        )
else:
    SHEETS_EXPORT_RESULT = {"written": False, "tables": []}
    print("ESCREVER_SHEETS = False — dry-run: nenhuma aba foi alterada.")


## 9. Contrato para o consumidor

Se você vai construir em cima desta base — Sheets, Lovable ou qualquer outra coisa — leia esta
seção. Ela é o que impede uma leitura errada dos números.

### Como as tabelas se ligam

```
dim_mp_fornecimento (fabric_sku_id)
   │ fabric_id                          │ fabric_sku_id
   ▼                                    ▼
fact_sku_bom (sku × fabric_id)     fact_mp_lt_realizado (fabric_order_id)
   │ sku
   ▼
fact_sku_economics (sku)
                                   mart_produto_mp (product_name) ← derivada, não junte por id
```

- `dim_mp_fornecimento` → `fact_sku_bom` por **`fabric_id`** (1 tecido : N cotações).
- `dim_mp_fornecimento` → `fact_mp_lt_realizado` por **`fabric_sku_id`**.
- `fact_sku_bom` → `fact_sku_economics` por **`sku`**. Cuidado: `fact_sku_bom` tem N linhas por
  SKU (uma por tecido) e `fact_sku_economics` tem 1. **Agregue antes de juntar**, ou o custo
  será contado várias vezes.
- `mart_produto_mp` é agregado por nome de produto e **não expõe `article_id`** — a
  normalização de nome funde vários ids, então nenhum deles representaria o grupo.

### Cinco coisas que fazem alguém errar

1. **Custo de MP não é CMV.** `custo_mp_*` é matéria-prima isolada; `cmv_unitario` é custo
   total. Um não valida o outro. `pct_mp_no_cmv` mostra a relação entre eles.
2. **A base é snapshot, não série histórica.** Nenhuma tabela Muninn versiona preço ou status.
   Rodar o notebook amanhã sobrescreve o retrato de hoje. **Não dá para calcular variação de
   custo de MP no tempo com esta base** — só `cmv_data_ref` tem eixo temporal real. Quem
   precisar de tendência tem que acumular snapshots por fora.
3. **`custo_min_fabric` e `custo_max_fabric` são a faixa entre malharias**, não mínimo e máximo
   histórico. Quando `n_malharias = 1`, os dois são iguais e o tecido tem **fonte única** — é
   informação de risco de fornecimento, não ruído.
4. **Atraso de MP se mede contra `data_faturamento_estimada_original`.** A coluna repactuada
   existe para diagnóstico. Trocar uma pela outra faz o atraso sumir.
5. **NULL é ausência de cadastro, não zero.** `gramatura_g_m2` e `largura_cm` estão vazias em
   100% das linhas por falta de fonte. `lt_malharia_cadastrado_dias` está vazio onde a malharia
   não tem lead time cadastrado. Preencher com zero inventaria dado.

### O que está fora do escopo

Confecção inteira: lead time cadastrado, lead time realizado e recebimento de MP na ordem de
produção. `product_production_parameters` e `muninn_production_orders` seguem disponíveis para
uma v2, se o escopo mudar.

Também fora: `muninn_products_articles` (BOM em grão de produto, para planejamento). Se entrar
depois, o campo deve se chamar `consumo_produto_estimado` — nunca `consumption` — para não ser
somado com `consumo_sku`.

### Para destravar os gaps

| Gap | O que falta | Onde buscar |
|---|---|---|
| Gramatura e largura | Não existe no data lake | Cadastro Muninn ou planilha do time de produto |
| Série histórica de custo de MP | Tabela `_history` para `fabric_skus` | Time de dados |
| LT de malharia cadastrado incompleto | Cadastro em `muninn_articles_knitting_factories` | Time de cadastro / suprimentos |


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=62b1671d-dc82-4747-9bbb-b134a69a3491' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>